###1. Импорт библиотек и чтение Excel

In [9]:
import pandas as pd
import numpy as np

# Путь к файлу
file_path = 'кейс 40.xlsx'

# Чтение всех листов
df_product = pd.read_excel(file_path, sheet_name='Product')
df_region  = pd.read_excel(file_path, sheet_name='Region')
df_sales   = pd.read_excel(file_path, sheet_name='Sales')
df_state   = pd.read_excel(file_path, sheet_name='State')

print("Product:", df_product.shape)
print("Region:", df_region.shape)
print("Sales:", df_sales.shape)
print("State:", df_state.shape)

Product: (20, 9)
Region: (6, 2)
Sales: (377741, 9)
State: (51, 4)


2. Первичная очистка и преобразование типов


In [10]:
# --- Sales ---
df_sales['OrderDate'] = pd.to_datetime(df_sales['OrderDate'])
df_sales['ShipDate']  = pd.to_datetime(df_sales['ShipDate'])

# Заполняем пустые промокоды
df_sales['PromotionCode'] = df_sales['PromotionCode'].fillna('No Promotion')

# Проверка, что Quantity и UnitPrice числовые
df_sales['Quantity']       = pd.to_numeric(df_sales['Quantity'], errors='coerce')
df_sales['UnitPrice']      = pd.to_numeric(df_sales['UnitPrice'], errors='coerce')
df_sales['DiscountAmount'] = pd.to_numeric(df_sales['DiscountAmount'], errors='coerce')

# --- Product ---
df_product['RetailPrice'] = pd.to_numeric(df_product['RetailPrice'], errors='coerce')
df_product['Channels']    = pd.to_numeric(df_product['Channels'], errors='coerce')

# --- State ---
df_state['RegionID'] = pd.to_numeric(df_state['RegionID'], errors='coerce')

3. Создание производных полей в Sales

In [11]:
# Выручка
df_sales['GrossRevenue'] = df_sales['Quantity'] * df_sales['UnitPrice']
df_sales['NetRevenue']   = df_sales['GrossRevenue'] - df_sales['DiscountAmount']

# Временные признаки
df_sales['OrderYear']    = df_sales['OrderDate'].dt.year
df_sales['OrderMonth']   = df_sales['OrderDate'].dt.to_period('M')
df_sales['OrderQuarter'] = df_sales['OrderDate'].dt.to_period('Q')
df_sales['OrderWeekday'] = df_sales['OrderDate'].dt.day_name()

# Срок отгрузки
df_sales['ShipDays'] = (df_sales['ShipDate'] - df_sales['OrderDate']).dt.days

4. Объединённый датасет продаж (fact_sales)

In [12]:
# География: State + Region
df_geo = df_state.merge(df_region, on='RegionID', how='left')

# Основной факт
df_fact = (
    df_sales
    .merge(df_product, on='ProductID', how='left')
    .merge(df_geo, left_on='CustomerStateID', right_on='StateID', how='left')
)

# Переименуем некоторые колонки для удобства
df_fact = df_fact.rename(columns={
    'StateName': 'CustomerState',
    'RegionName': 'CustomerRegion'
})

print(df_fact.head())
print(df_fact.columns.tolist())

  OrderNumber  OrderDate   ShipDate  CustomerStateID  ProductID  Quantity  \
0  TT00017434 2017-01-01 2017-01-09               14          3         1   
1  TT00017445 2017-01-02 2017-01-04               13          3         1   
2  TT00017456 2017-01-06 2017-01-09               35          3         1   
3  TT00017467 2017-01-08 2017-01-11                3          3         1   
4  TT00017478 2017-01-08 2017-01-10                5          3         1   

   UnitPrice  DiscountAmount PromotionCode  GrossRevenue  ...  ItemGroup  \
0      69.95             0.0  No Promotion         69.95  ...   Airplane   
1      69.95             0.0  No Promotion         69.95  ...   Airplane   
2      69.95             0.0  No Promotion         69.95  ...   Airplane   
3      69.95             0.0  No Promotion         69.95  ...   Airplane   
4      69.95             0.0  No Promotion         69.95  ...   Airplane   

   KitType Channels Demographic RetailPrice  StateID StateCode CustomerState  \


5. Создание календарного датасета (dim_date)

In [13]:
min_date = df_sales['OrderDate'].min()
max_date = df_sales['OrderDate'].max()

df_date = pd.DataFrame({
    'Date': pd.date_range(min_date, max_date, freq='D')
})
df_date['Year']      = df_date['Date'].dt.year
df_date['Quarter']   = df_date['Date'].dt.quarter
df_date['Month']     = df_date['Date'].dt.month
df_date['MonthName'] = df_date['Date'].dt.month_name()
df_date['Week']      = df_date['Date'].dt.isocalendar().week
df_date['Weekday']   = df_date['Date'].dt.day_name()
df_date['IsWeekend'] = df_date['Date'].dt.weekday >= 5

6. Аналитические витрины (датасеты для отчётов)

6.1 Продажи по продуктам

In [14]:
sales_by_product = (
    df_fact
    .groupby(['ProductID', 'ProductName', 'ProductCategory', 'ItemGroup', 'KitType', 'Demographic'])
    .agg(
        Orders       = ('OrderNumber', 'nunique'),
        Units        = ('Quantity', 'sum'),
        GrossRevenue = ('GrossRevenue', 'sum'),
        NetRevenue   = ('NetRevenue', 'sum'),
        Discount     = ('DiscountAmount', 'sum')
    )
    .reset_index()
)

6.2. Продажи по регионам

In [15]:
sales_by_region = (
    df_fact
    .groupby(['CustomerRegion'])
    .agg(
        Orders       = ('OrderNumber', 'nunique'),
        Units        = ('Quantity', 'sum'),
        GrossRevenue = ('GrossRevenue', 'sum'),
        NetRevenue   = ('NetRevenue', 'sum'),
        Discount     = ('DiscountAmount', 'sum')
    )
    .reset_index()
)

6.3. Продажи по штатам

In [16]:
sales_by_state = (
    df_fact
    .groupby(['StateCode', 'CustomerState', 'CustomerRegion'])
    .agg(
        Orders       = ('OrderNumber', 'nunique'),
        Units        = ('Quantity', 'sum'),
        NetRevenue   = ('NetRevenue', 'sum')
    )
    .reset_index()
)

6.4. Продажи по месяцам

In [17]:
sales_by_month = (
    df_fact
    .groupby(df_fact['OrderDate'].dt.to_period('M'))
    .agg(
        Orders       = ('OrderNumber', 'nunique'),
        Units        = ('Quantity', 'sum'),
        GrossRevenue = ('GrossRevenue', 'sum'),
        NetRevenue   = ('NetRevenue', 'sum'),
        Discount     = ('DiscountAmount', 'sum')
    )
    .reset_index()
    .rename(columns={'OrderDate': 'Month'})
)

6.5. Эффективность промо-акций

In [18]:
sales_by_promo = (
    df_fact
    .groupby(['PromotionCode'])
    .agg(
        Orders       = ('OrderNumber', 'nunique'),
        Units        = ('Quantity', 'sum'),
        NetRevenue   = ('NetRevenue', 'sum'),
        Discount     = ('DiscountAmount', 'sum')
    )
    .reset_index()
)

6.6. Сравнение UnitPrice с RetailPrice

In [19]:
price_analysis = (
    df_fact
    .groupby(['ProductID', 'ProductName'])
    .agg(
        AvgUnitPrice  = ('UnitPrice', 'mean'),
        RetailPrice   = ('RetailPrice', 'first'),
        AvgDiscount   = ('DiscountAmount', 'mean'),
        Units         = ('Quantity', 'sum')
    )
    .reset_index()
)
price_analysis['PriceDiff'] = price_analysis['AvgUnitPrice'] - price_analysis['RetailPrice']
price_analysis['PriceDiffPct'] = price_analysis['PriceDiff'] / price_analysis['RetailPrice'] * 100

7. Сохранение датасетов

In [ ]:
# В CSV
df_product.to_csv('dim_product.csv', index=False)
df_region.to_csv('dim_region.csv', index=False)
df_state.to_csv('dim_state.csv', index=False)
df_date.to_csv('dim_date.csv', index=False)
df_fact.to_csv('fact_sales.csv', index=False)

sales_by_product.to_csv('sales_by_product.csv', index=False)
sales_by_region.to_csv('sales_by_region.csv', index=False)
sales_by_state.to_csv('sales_by_state.csv', index=False)
sales_by_month.to_csv('sales_by_month.csv', index=False)
sales_by_promo.to_csv('sales_by_promo.csv', index=False)
price_analysis.to_csv('price_analysis.csv', index=False)

# Или в один Excel
with pd.ExcelWriter('analytics_datasets.xlsx') as writer:
    df_product.to_excel(writer, sheet_name='dim_product', index=False)
    df_region.to_excel(writer, sheet_name='dim_region', index=False)
    df_state.to_excel(writer, sheet_name='dim_state', index=False)
    df_date.to_excel(writer, sheet_name='dim_date', index=False)
    df_fact.to_excel(writer, sheet_name='fact_sales', index=False)
    sales_by_product.to_excel(writer, sheet_name='sales_by_product', index=False)
    sales_by_region.to_excel(writer, sheet_name='sales_by_region', index=False)
    sales_by_state.to_excel(writer, sheet_name='sales_by_state', index=False)
    sales_by_month.to_excel(writer, sheet_name='sales_by_month', index=False)
    sales_by_promo.to_excel(writer, sheet_name='sales_by_promo', index=False)
    price_analysis.to_excel(writer, sheet_name='price_analysis', index=False)